# FC-Siam-diff on d2c_v3

Training recipe from Daudt et al. (ICIP 2018), `fully-convolutional-change-detection.ipynb`. Model: upstream `siamunet_diff.py`, imported unmodified. Sections follow `spec.md`.

## Config

In [ ]:
from pathlib import Path

DATA_ROOT = Path('../../data/processed/d2c_v3').resolve()
RUNS_DIR = Path('runs')
INDEX = DATA_ROOT / 'master_dataset_with_benign.json'
SPLITS = DATA_ROOT / 'splits_85_15.json'
MODEL_CODE = Path('.')         # folder holding upstream siamunet_diff.py (not committed: no license)

INPUT_MODE = 'diff_squares'    # 'diff_squares' | 'random_squares' | 'all_squares' | 'whole_page'
SQUARE = 96                    # square side in px (notebook PATCH_SIDE)
SQUARES_PER_PAIR = 16
RANDOM_MIX = 0.0               # diff_squares: share of squares placed at random
STRIDE = SQUARE // 2 - 1       # all_squares: step between squares (notebook TRAIN_STRIDE)
PAGE_SIZE = (1024, 512)        # whole_page: (H, W) every page is resized to

BATCH_SIZE = 32
N_EPOCHS = 50
FP_MODIFIER = 10
TYPE_WEIGHT_POWER = 0.5        # 0 = every type weighted the same; 1 = each type carries the same total weight
TYPE_WEIGHT_OVERRIDE = {}      # e.g. {'border_radius': 20}; replaces the computed weight for that type
WEIGHT_DECAY = 1e-4
LR_GAMMA = 0.95
AUG_FLIP = True
AUG_ROT = False

GATE_WITH_DIFF = True          # identical pixels can never be flagged

SCORE_RULE = 'top_n'           # 'top_n' | 'max' | 'flagged_conf'
OP_RULE = 'share'              # 'share' | 'weighted' | 'region'
BOX_RULE = 'all'               # 'all' | 'largest'
POINT_RULE = 'peak'            # 'peak' | 'topmost'
TOP_N = 50                     # smallest regression in the dataset changes 60 px
PIXEL_THRESHOLD = 0.5

NUM_WORKERS = 12
SEED = 0
BF16 = True
RESUME = None                  # e.g. 'runs/<id>/ckpt_last.pt'

PAIRS_PER_LOAD = {'all_squares': 1, 'whole_page': BATCH_SIZE}.get(INPUT_MODE, BATCH_SIZE // SQUARES_PER_PAIR)

In [ ]:
import json, random, sys, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy import ndimage
from tqdm import tqdm

sys.path.insert(0, str(MODEL_CODE))
from siamunet_diff import SiamUnet_diff

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
amp = dict(device_type=device.type, dtype=torch.bfloat16, enabled=BF16 and device.type == 'cuda')
print(device)

## Split by page (`splits_85_15.json`)

In [ ]:
rows = json.loads(INDEX.read_text())
split = {k: set(v) for k, v in json.loads(SPLITS.read_text()).items()}
assert not split['train'] & split['val']

train_rows = [r for r in rows if r['parent_id'] in split['train']]
val_rows = [r for r in rows if r['parent_id'] in split['val']]
assert len(train_rows) + len(val_rows) == len(rows)

# Failed Firefox renders: the candidate screenshot is a single flat colour (scan of all 7,755 images); kept in val, which must cover every pair
BLANK_RENDERS = {'d2c_324_firefox', 'd2c_183_firefox'}
train_rows = [r for r in train_rows if r['sample_id'] not in BLANK_RENDERS]

for name, rs in (('train', train_rows), ('val', val_rows)):
    print(name, len({r['parent_id'] for r in rs}), 'pages', dict(Counter(r['label'] for r in rs)))

## 2.1 Training target: 14 classes

Class 0 is "not a regression"; classes 1–13 are the operators. In a regression pair, every pixel that differs gets the pair's operator. Everything else is 0, including every difference in `benign` and `no_change` pairs.

In [ ]:
FAMILY = {r['operator']: r['family'] for r in rows if r['label'] == 'regression'}
OPS = sorted(FAMILY)
CLASSES = ['no_regression'] + OPS
assert len(OPS) == 13

def load_pair(row):
    pa, pb = load_images(row)
    diff = (pa != pb).any(2)
    cls = CLASSES.index(row['operator']) if row['label'] == 'regression' else 0
    return to_tensor(pa), to_tensor(pb), torch.from_numpy(diff * cls), torch.from_numpy(diff)

## 2.2 Alignment

Decode as RGB (drops the alpha channel only Firefox/WebKit shots have) and pad both to the larger size with white at the top-left. Uses only the image paths, never the label.

In [ ]:
def to_tensor(I):
    I = I.astype(np.float32)
    return torch.from_numpy(((I - I.mean()) / (I.std() or 1)).transpose(2, 0, 1))  # a blank image has std 0

def load_images(row):
    a = np.asarray(Image.open(DATA_ROOT / row['reference_path']).convert('RGB'))
    b = np.asarray(Image.open(DATA_ROOT / row['candidate_path']).convert('RGB'))
    H, W = max(a.shape[0], b.shape[0]), max(a.shape[1], b.shape[1])
    pa = np.full((H, W, 3), 255, np.uint8); pa[:a.shape[0], :a.shape[1]] = a
    pb = np.full((H, W, 3), 255, np.uint8); pb[:b.shape[0], :b.shape[1]] = b
    return pa, pb

In [ ]:
# Check: diff equals the dataset's changed_px for every pair; the target marks only regression pixels, with the pair's operator
for r in random.sample(train_rows, 30):
    *_, t, d = load_pair(r)
    assert d.sum() == r['changed_px']
    assert (t > 0).sum() == (r['changed_px'] if r['label'] == 'regression' else 0)
    assert set(t.unique().tolist()) <= {0, CLASSES.index(r['operator']) if r['operator'] else 0}
print('targets consistent with index')

## 2.3 What the network sees during training

`diff_squares`: every square is centred on a pixel where the two screenshots differ (raw diff, not the label), so benign noise and real changes both get squares. `no_change` pairs fall back to random squares.

In [ ]:
def random_corners(H, W, n):
    ys = torch.randint(0, H - SQUARE + 1, (n,))
    xs = torch.randint(0, W - SQUARE + 1, (n,))
    return list(zip(ys.tolist(), xs.tolist()))

# Centre each square on a random differing pixel, shifted to stay inside the page
def diff_corners(diff, n):
    H, W = diff.shape
    yx = diff.nonzero()
    if len(yx) == 0:
        return random_corners(H, W, n)
    c = yx[torch.randint(len(yx), (n,))] - SQUARE // 2
    return list(zip(c[:, 0].clamp(0, H - SQUARE).tolist(), c[:, 1].clamp(0, W - SQUARE).tolist()))

def square_corners(diff):
    H, W = diff.shape
    if INPUT_MODE == 'diff_squares':
        n_rand = round(RANDOM_MIX * SQUARES_PER_PAIR)
        return diff_corners(diff, SQUARES_PER_PAIR - n_rand) + random_corners(H, W, n_rand)
    if INPUT_MODE == 'random_squares':
        return random_corners(H, W, SQUARES_PER_PAIR)
    return [(y, x) for y in range(0, H - SQUARE + 1, STRIDE) for x in range(0, W - SQUARE + 1, STRIDE)]

shrink = lambda I: F.interpolate(I[None], PAGE_SIZE, mode='area')
# whole_page: max-pooling keeps every marked pixel after shrinking; a pair has one class, so it holds for the target too
shrink_map = lambda m: F.adaptive_max_pool2d(m[None, None].float(), PAGE_SIZE)[0].to(m.dtype)

def to_inputs(I1, I2, t, diff):
    if INPUT_MODE == 'whole_page':
        return shrink(I1), shrink(I2), shrink_map(t)
    cs = square_corners(diff)
    crop = lambda T: torch.stack([T[..., y:y + SQUARE, x:x + SQUARE] for y, x in cs])
    return crop(I1), crop(I2), crop(t)

def augment(I1, I2, t):
    if AUG_FLIP and torch.rand(1) < 0.5:
        I1, I2, t = I1.flip(-1), I2.flip(-1), t.flip(-1)
    if AUG_ROT:
        k = int(torch.randint(0, 4, (1,)))
        I1, I2, t = (T.rot90(k, (-2, -1)) for T in (I1, I2, t))
    return I1, I2, t

In [ ]:
class PairDataset(Dataset):
    def __init__(self, rows, mode):  # 'train' | 'val' | 'predict'
        self.rows, self.mode = rows, mode

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        if self.mode == 'predict':
            pa, pb = load_images(self.rows[i])
            return to_tensor(pa), to_tensor(pb), torch.from_numpy((pa != pb).any(2)), i
        I1, I2, t, diff = load_pair(self.rows[i])
        if self.mode == 'train':
            return augment(*to_inputs(I1, I2, t, diff))
        return I1, I2, t, diff, i

def cat_collate(batch):
    return tuple(torch.cat(t) for t in zip(*batch))

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(PairDataset(train_rows, 'train'), batch_size=PAIRS_PER_LOAD, shuffle=True, collate_fn=cat_collate, **loader_kw)
val_loader = DataLoader(PairDataset(val_rows, 'val'), batch_size=1, **loader_kw)

## 2.4 Training recipe: network, per-type loss weights, optimizer

In [ ]:
n_pix = sum(max(r['size_ref'][0], r['size_cand'][0]) * max(r['size_ref'][1], r['size_cand'][1]) for r in train_rows)
op_pix = Counter()
for r in train_rows:
    if r['label'] == 'regression':
        op_pix[r['operator']] += r['changed_px']
true_pix = sum(op_pix.values())

# Notebook weight formula; the "change" weight is shifted toward rare types by TYPE_WEIGHT_POWER, keeping its total over all regression pixels fixed
w_change = 2 * (n_pix - true_pix) / n_pix
w = {o: (true_pix / op_pix[o]) ** TYPE_WEIGHT_POWER for o in OPS}
norm = w_change * true_pix / sum(w[o] * op_pix[o] for o in OPS)
w = {o: TYPE_WEIGHT_OVERRIDE.get(o, w[o] * norm) for o in OPS}
weights = torch.tensor([FP_MODIFIER * 2 * true_pix / n_pix] + [w[o] for o in OPS], device=device)
print(dict(zip(CLASSES, weights.tolist())))

net = SiamUnet_diff(3, len(CLASSES)).to(device)
criterion = nn.NLLLoss(weight=weights)
optimizer = torch.optim.Adam(net.parameters(), weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, LR_GAMMA)
print('Number of trainable parameters:', sum(p.numel() for p in net.parameters() if p.requires_grad))

## 2.5 Diff veto and evaluation (val pages, whole page at once)

With `GATE_WITH_DIFF`, pixels that are identical in both screenshots are forced to "not a regression". Pixel metrics count "regression vs not"; `pixel_type_accuracy` is the share of correctly flagged pixels that also get the right type.

In [ ]:
def div(a, b):
    return a / b if b else 0.0

def kappa(tp, tn, fp, fn):
    N = tp + tn + fp + fn
    p0 = (tp + tn) / N
    pe = ((tp + fp) * (tp + fn) + (tn + fp) * (tn + fn)) / (N * N)
    return div(p0 - pe, 1 - pe)

# Move one pair to the device, shrink it in whole_page mode, and return the factor back to original pixels
def on_page(I1, I2, *maps):
    I1, I2, *maps = (T[0].to(device) for T in (I1, I2, *maps))
    H, W = maps[0].shape
    if INPUT_MODE == 'whole_page':
        I1, I2, maps = shrink(I1)[0], shrink(I2)[0], [shrink_map(m)[0] for m in maps]
    return (I1, I2, *maps, (W / maps[0].shape[1], H / maps[0].shape[0]))

@torch.no_grad()
def predict(I1, I2, diff):
    with torch.autocast(**amp):
        logp = net(I1[None], I2[None]).float()[0]
    p = logp.exp()
    if GATE_WITH_DIFF:
        p[1:] *= diff
        p[0] = 1 - p[1:].sum(0)
    return logp, p

@torch.no_grad()
def evaluate(loader, rows):
    net.eval()
    tp = tn = fp = fn = type_ok = 0
    tot_loss, recs = 0.0, {}
    for I1, I2, t, diff, i in tqdm(loader, leave=False):
        I1, I2, t, diff, scale = on_page(I1, I2, t, diff)
        logp, p = predict(I1, I2, diff)
        tot_loss += criterion(logp[None], t[None]).item() * t.numel()
        pred = p.argmax(0)
        pr, g = pred > 0, t > 0
        tp += (pr & g).sum().item(); tn += (~pr & ~g).sum().item()
        fp += (pr & ~g).sum().item(); fn += (~pr & g).sum().item()
        type_ok += (g & (pred == t)).sum().item()
        recs[int(i)] = page_record(p, scale)

    flagged = {i: recs[i]['box'] is not None for i in recs}
    is_reg = {i: rows[i]['label'] == 'regression' for i in recs}
    regs = [i for i in recs if is_reg[i]]
    itp = sum(flagged[i] and is_reg[i] for i in recs)
    ifp = sum(flagged[i] and not is_reg[i] for i in recs)
    ifn = sum(not flagged[i] and is_reg[i] for i in recs)
    prec, rec = div(tp, tp + fp), div(tp, tp + fn)
    iprec, irec = div(itp, itp + ifp), div(itp, itp + ifn)
    hit = lambda i: recs[i]['op_pred'] == rows[i]['operator']
    return {
        'val_loss': tot_loss / (tp + tn + fp + fn),
        'pixel_acc_nochange': div(tn, tn + fp), 'pixel_acc_change': rec,
        'pixel_precision': prec, 'pixel_recall': rec, 'pixel_f1': div(2 * prec * rec, prec + rec),
        'pixel_kappa': kappa(tp, tn, fp, fn), 'pixel_type_accuracy': div(type_ok, tp),
        'image_accuracy': sum(flagged[i] == is_reg[i] for i in recs) / len(recs),
        'image_precision': iprec, 'image_recall': irec, 'image_f1': div(2 * iprec * irec, iprec + irec),
        **{f'image_fpr_{lab}': float(np.mean([flagged[i] for i in recs if rows[i]['label'] == lab]))
           for lab in ('no_change', 'benign')},
        'type_accuracy': float(np.mean([hit(i) for i in regs])),
        'family_accuracy': float(np.mean([FAMILY[recs[i]['op_pred']] == rows[i]['family'] for i in regs])),
        'type_accuracy_by_op': {o: float(np.mean(h)) for o in OPS if (h := [hit(i) for i in regs if rows[i]['operator'] == o])},
    }

## 5. Output rules (spec 5.3)

Turn one page's 14 × H × W probabilities into a prediction-file line. A pixel's regression confidence is 1 − P(not a regression); it is flagged above `PIXEL_THRESHOLD`. Each field's rule is a config setting. Coordinates are in original screenshot pixels.

In [ ]:
def box_of(sl):
    return [sl[1].start, sl[0].start, sl[1].stop - sl[1].start, sl[0].stop - sl[0].start]

def page_record(p, scale=(1.0, 1.0)):
    reg = 1 - p[0]
    flagged = reg > PIXEL_THRESHOLD
    top = reg.flatten().topk(min(TOP_N, reg.numel())).values
    # flagged_conf: summed confidence of flagged pixels, relative to N pixels at full confidence
    score = {'top_n': top.mean(), 'max': top[0], 'flagged_conf': (reg[flagged].sum() / TOP_N).clamp(max=1)}[SCORE_RULE]
    box = point = None
    if flagged.any():
        lab, _ = ndimage.label(flagged.cpu().numpy())
        regions = ndimage.find_objects(lab)
        peak = divmod(int(reg.argmax()), reg.shape[1])
        if OP_RULE == 'weighted':
            votes = p[1:, flagged].sum(1)
        else:
            region = flagged if OP_RULE == 'share' else torch.from_numpy(lab == lab[peak]).to(p.device)
            votes = torch.bincount(p[1:, region].argmax(0), minlength=len(OPS)).float()
        if BOX_RULE == 'largest':
            box = box_of(regions[int(np.argmax(np.bincount(lab.ravel())[1:]))])
        else:
            ys, xs = flagged.nonzero(as_tuple=True)
            box = [int(xs.min()), int(ys.min()), int(xs.max() - xs.min()) + 1, int(ys.max() - ys.min()) + 1]
        if POINT_RULE == 'topmost':
            x, y, w, h = box_of(min(regions, key=lambda s: s[0].start))
            point = [x + w / 2, y + h / 2]
        else:
            point = [peak[1], peak[0]]
    else:
        votes = p[1:].sum((1, 2))
    first, second = votes.topk(2).indices.tolist()
    sx, sy = scale
    return {
        'score': float(score), 'op_pred': OPS[first],
        'box': box and [box[0] * sx, box[1] * sy, box[2] * sx, box[3] * sy],
        'point': point and [point[0] * sx, point[1] * sy],
        'op_second': OPS[second],
    }

## Train

In [ ]:
def save(path, state):
    tmp = path.with_suffix('.tmp')
    torch.save(state, tmp)
    tmp.replace(path)

start_epoch, best_f1 = 0, -1.0
if RESUME:
    ck = torch.load(RESUME, map_location=device)
    net.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scheduler.load_state_dict(ck['scheduler'])
    start_epoch, best_f1 = ck['epoch'] + 1, ck['best_f1']
    torch.set_rng_state(ck['rng'])
    run_dir = Path(RESUME).parent
else:
    run_dir = RUNS_DIR / time.strftime('%Y%m%d-%H%M%S')
    run_dir.mkdir(parents=True)
    (run_dir / 'config.json').write_text(json.dumps({k: v for k, v in globals().items() if k.isupper()}, default=str, indent=1))
print(run_dir)

In [ ]:
for epoch in range(start_epoch, N_EPOCHS):
    net.train()
    t0, losses = time.time(), []
    for I1, I2, t in tqdm(train_loader, desc=f'epoch {epoch + 1}/{N_EPOCHS}'):
        for s in range(0, len(t), BATCH_SIZE):
            a, b, y = (T[s:s + BATCH_SIZE].to(device, non_blocking=True) for T in (I1, I2, t))
            optimizer.zero_grad()
            with torch.autocast(**amp):
                out = net(a, b)
            loss = criterion(out.float(), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
    scheduler.step()

    metrics = {'epoch': epoch + 1, 'train_loss': float(np.mean(losses)), **evaluate(val_loader, val_rows), 'seconds': round(time.time() - t0)}
    print(metrics)
    with open(run_dir / 'metrics.jsonl', 'a') as f:
        f.write(json.dumps(metrics) + '\n')

    is_best = metrics['pixel_f1'] > best_f1
    best_f1 = max(best_f1, metrics['pixel_f1'])
    state = dict(model=net.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                 epoch=epoch, best_f1=best_f1, rng=torch.get_rng_state())
    save(run_dir / 'ckpt_last.pt', state)
    if is_best:
        save(run_dir / 'ckpt_best.pt', state)

## 5. Prediction file (spec 5.1, 5.4)

Best checkpoint, every validation pair once, images only (no label is read). Score with the team's `scripts/score_predictions.py`.

In [ ]:
def write_predictions(rows, path):
    net.eval()
    loader = DataLoader(PairDataset(rows, 'predict'), batch_size=1, **loader_kw)
    with open(path, 'w') as f:
        for I1, I2, diff, i in tqdm(loader):
            I1, I2, diff, scale = on_page(I1, I2, diff)
            rec = page_record(predict(I1, I2, diff)[1], scale)
            line = {'sample_id': rows[int(i)]['sample_id'], **rec, 'model': f'fc_siam_diff/{run_dir.name}', 'op_family': FAMILY[rec['op_pred']]}
            f.write(json.dumps(line) + '\n')

net.load_state_dict(torch.load(run_dir / 'ckpt_best.pt', map_location=device)['model'])
pred_path = run_dir / 'fc_siam_diff_val.jsonl'
write_predictions(val_rows, pred_path)
ids = [json.loads(l)['sample_id'] for l in pred_path.read_text().splitlines()]
assert sorted(ids) == sorted(r['sample_id'] for r in val_rows)
print(pred_path, len(ids), 'predictions')